# LightGBM Competition CV Solution

This notebook trains a stronger LightGBM baseline for the QRT electricity price challenge. It uses day-grouped cross-validation, compact feature engineering, early stopping, and averaged fold/seed predictions for the final submission.

## Setup

Run the environment from the project root with `uv sync`, then open this notebook using the `.venv` kernel or run the cells through `uv run` tooling.

In [ ]:
from pathlib import Path
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"
SUBMISSION_DIR = PROJECT_ROOT / "submissions"
SUBMISSION_DIR.mkdir(exist_ok=True)

TRAIN_X_PATH = DATA_DIR / "X_train.csv"
TRAIN_Y_PATH = DATA_DIR / "y_train.csv"
TEST_X_PATH = DATA_DIR / "X_test_final.csv"
SUBMISSION_PATH = SUBMISSION_DIR / "lightgbm_cv_submission.csv"
OOF_PATH = SUBMISSION_DIR / "lightgbm_oof_predictions.csv"

RANDOM_SEEDS = [13, 37, 71]
N_SPLITS = 5

In [ ]:
X_train = pd.read_csv(TRAIN_X_PATH)
y_train = pd.read_csv(TRAIN_Y_PATH)
X_test = pd.read_csv(TEST_X_PATH)

train = X_train.merge(y_train, on="ID", how="inner", validate="one_to_one")

assert len(train) == 1494, f"Expected 1494 training rows, found {len(train)}"
assert len(X_test) == 654, f"Expected 654 test rows, found {len(X_test)}"
assert train["TARGET"].notna().all(), "Training target contains missing values"

print(f"train: {train.shape}")
print(f"test:  {X_test.shape}")
print(f"train days: {train['DAY_ID'].nunique()} | test days: {X_test['DAY_ID'].nunique()}")
print(f"countries: {train['COUNTRY'].value_counts().to_dict()}")

## Feature Engineering

The features stay intentionally compact: country encoding, FR-DE paired spreads, import/export balances, exchange balance, and missingness indicators. LightGBM handles remaining numeric missing values natively.

In [ ]:
def add_features(df: pd.DataFrame, missing_indicator_columns: list[str]) -> pd.DataFrame:
    out = df.copy()

    out["COUNTRY"] = out["COUNTRY"].astype("category")
    out["IS_FR"] = (out["COUNTRY"].astype(str) == "FR").astype("int8")

    # FR minus DE spreads for paired country measurements.
    paired_suffixes = sorted({
        col[3:]
        for col in out.columns
        if col.startswith("FR_") and f"DE_{col[3:]}" in out.columns
    })
    for suffix in paired_suffixes:
        fr_col = f"FR_{suffix}"
        de_col = f"DE_{suffix}"
        out[f"FR_DE_{suffix}_SPREAD"] = out[fr_col] - out[de_col]
        out[f"ACTIVE_{suffix}"] = np.where(out["IS_FR"].eq(1), out[fr_col], out[de_col])

    out["DE_IMPORT_EXPORT_BALANCE"] = out["DE_NET_IMPORT"] - out["DE_NET_EXPORT"]
    out["FR_IMPORT_EXPORT_BALANCE"] = out["FR_NET_IMPORT"] - out["FR_NET_EXPORT"]
    out["ACTIVE_IMPORT_EXPORT_BALANCE"] = np.where(
        out["IS_FR"].eq(1),
        out["FR_IMPORT_EXPORT_BALANCE"],
        out["DE_IMPORT_EXPORT_BALANCE"],
    )
    out["EXCHANGE_BALANCE"] = out["FR_DE_EXCHANGE"] - out["DE_FR_EXCHANGE"]

    for col in missing_indicator_columns:
        out[f"{col}_MISSING"] = out[col].isna().astype("int8")

    return out

combined_features = pd.concat(
    [X_train.drop(columns=["ID"]), X_test.drop(columns=["ID"])],
    axis=0,
    ignore_index=True,
)
missing_indicator_columns = sorted(
    col for col in combined_features.columns if combined_features[col].isna().any()
)

train_fe = add_features(train.drop(columns=["TARGET"]), missing_indicator_columns)
test_fe = add_features(X_test, missing_indicator_columns)

feature_columns = [col for col in train_fe.columns if col != "ID"]
X = train_fe[feature_columns]
y = train["TARGET"].astype(float)
X_test_model = test_fe[feature_columns]
groups = train_fe["DAY_ID"]

assert list(X.columns) == list(X_test_model.columns)
print(f"features: {len(feature_columns)}")
print(f"missing indicators: {missing_indicator_columns}")

## Cross-Validation

Validation is grouped by `DAY_ID`, so rows from the same day never appear in both the training and validation parts of a fold.

In [ ]:
def spearman_score(y_true, y_pred) -> float:
    corr = spearmanr(y_true, y_pred).correlation
    return 0.0 if np.isnan(corr) else float(corr)


def spearman_lgb_metric(y_true, y_pred):
    return "spearman", spearman_score(y_true, y_pred), True


base_params = {
    "objective": "regression",
    "metric": "None",
    "boosting_type": "gbdt",
    "learning_rate": 0.02,
    "n_estimators": 5000,
    "num_leaves": 15,
    "max_depth": 4,
    "min_child_samples": 20,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "force_col_wise": True,
    "deterministic": True,
    "verbosity": -1,
    "n_jobs": -1,
}

cv = GroupKFold(n_splits=N_SPLITS)
oof_predictions = np.zeros(len(X), dtype=float)
test_predictions = np.zeros(len(X_test_model), dtype=float)
fold_scores = []
models = []

for seed in RANDOM_SEEDS:
    print(f"\nSeed {seed}")
    seed_test_predictions = np.zeros(len(X_test_model), dtype=float)

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y, groups=groups), start=1):
        X_tr, X_va = X.iloc[train_idx], X.iloc[valid_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[valid_idx]

        model = lgb.LGBMRegressor(**base_params, random_state=seed)
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric=spearman_lgb_metric,
            categorical_feature=["COUNTRY"],
            callbacks=[
                lgb.early_stopping(stopping_rounds=150, first_metric_only=True, verbose=False),
                lgb.log_evaluation(period=0),
            ],
        )

        valid_pred = model.predict(X_va, num_iteration=model.best_iteration_)
        fold_score = spearman_score(y_va, valid_pred)
        fold_scores.append({"seed": seed, "fold": fold, "spearman": fold_score, "best_iteration": model.best_iteration_})
        print(f"  fold {fold}: spearman={fold_score:.5f}, best_iteration={model.best_iteration_}")

        oof_predictions[valid_idx] += valid_pred / len(RANDOM_SEEDS)
        seed_test_predictions += model.predict(X_test_model, num_iteration=model.best_iteration_) / N_SPLITS
        models.append(model)

    test_predictions += seed_test_predictions / len(RANDOM_SEEDS)

overall_oof_score = spearman_score(y, oof_predictions)
fold_scores_df = pd.DataFrame(fold_scores)

print("\nFold score summary:")
print(fold_scores_df.groupby("seed")["spearman"].agg(["mean", "std"]))
print(f"\nOverall OOF Spearman: {overall_oof_score:.5f}")

## Diagnostics

In [ ]:
importance = pd.DataFrame({
    "feature": feature_columns,
    "importance_gain": np.mean([m.booster_.feature_importance(importance_type="gain") for m in models], axis=0),
    "importance_split": np.mean([m.booster_.feature_importance(importance_type="split") for m in models], axis=0),
}).sort_values("importance_gain", ascending=False)

importance.head(25)

In [ ]:
oof = train[["ID", "DAY_ID", "COUNTRY", "TARGET"]].copy()
oof["PREDICTION"] = oof_predictions
oof.to_csv(OOF_PATH, index=False)

print(f"Saved OOF predictions to {OOF_PATH.relative_to(PROJECT_ROOT)}")
oof.head()

## Generate Submission

In [ ]:
submission = X_test[["ID"]].copy()
submission["TARGET"] = test_predictions

expected_ids = pd.read_csv(TEST_X_PATH, usecols=["ID"])
assert submission.shape == (654, 2), f"Unexpected submission shape: {submission.shape}"
assert list(submission.columns) == ["ID", "TARGET"]
assert submission["ID"].equals(expected_ids["ID"]), "Submission IDs do not match X_test_final.csv"
assert submission["TARGET"].notna().all(), "Submission contains missing predictions"

submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Saved submission to {SUBMISSION_PATH.relative_to(PROJECT_ROOT)}")
submission.head()